# WAN 2.2 I2V LightX2V — Google Colab セットアップ

**推奨ランタイム: A100 GPU**（97フレーム生成には約30GB VRAM必要）

実行順序: セル1 → セル2 → セル3 → セル4

In [ ]:
# ============================================================
# セル1: 環境セットアップ
# ============================================================
import os, subprocess, sys
from google.colab import drive

# ── GPU 確認 ──────────────────────────────────────────────
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                         "--format=csv,noheader"], capture_output=True, text=True)
print(f"GPU: {result.stdout.strip()}")
vram_mb = int(result.stdout.strip().split(",")[1].strip().replace(" MiB", ""))
if vram_mb < 25000:
    print("\n⚠️  警告: VRAMが25GB未満です。")
    print("   97フレーム生成にはA100(40GB)を推奨します。")
    print("   ランタイム → ランタイムのタイプを変更 → A100 を選択してください。")
else:
    print("✅ VRAM 十分です")

# ── Google Drive マウント ──────────────────────────────────
print("\nGoogle Drive をマウント中...")
drive.mount("/content/drive")

# ── ComfyUI インストール / 最新化 ─────────────────────────
if not os.path.exists("/content/ComfyUI"):
    print("ComfyUI をインストール中...")
    subprocess.run(
        "git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI",
        shell=True, check=True
    )
else:
    print("ComfyUI を最新版に更新中...")
    subprocess.run("git -C /content/ComfyUI pull", shell=True, check=True)
    print("✅ ComfyUI 更新完了")

# ── 依存ライブラリのインストール ──────────────────────────
print("依存ライブラリをインストール中...")
subprocess.run(
    f"{sys.executable} -m pip install -q -r /content/ComfyUI/requirements.txt",
    shell=True, check=True
)
# VRAM効率化・高速化ライブラリ
subprocess.run(
    f"{sys.executable} -m pip install -q accelerate",
    shell=True, check=True
)

# ── Drive 上にモデル保存フォルダを作成 ─────────────────────
DRIVE_BASE = "/content/drive/MyDrive/ComfyUI_models"
for d in ["diffusion_models", "clip", "vae", "clip_vision", "loras"]:
    os.makedirs(f"{DRIVE_BASE}/{d}", exist_ok=True)

# ── ComfyUI に Drive のパスを登録 ─────────────────────────
yaml = f"""wan_drive:
    base_path: {DRIVE_BASE}
    diffusion_models: diffusion_models/
    clip: clip/
    vae: vae/
    loras: loras/
    clip_vision: clip_vision/
"""
with open("/content/ComfyUI/extra_model_paths.yaml", "w") as f:
    f.write(yaml)

print(f"\n✅ セル1 完了")
print(f"モデル保存先: {DRIVE_BASE}")

In [ ]:
# ============================================================
# セル2: モデルダウンロード（Drive キャッシュ対応）
# ============================================================
import os, subprocess

DRIVE_BASE = "/content/drive/MyDrive/ComfyUI_models"
BASE = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main"

FILES = [
    (f"{BASE}/split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp16.safetensors",
     f"{DRIVE_BASE}/diffusion_models/wan2.2_i2v_high_noise_14B_fp16.safetensors"),
    (f"{BASE}/split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp16.safetensors",
     f"{DRIVE_BASE}/diffusion_models/wan2.2_i2v_low_noise_14B_fp16.safetensors"),
    (f"{BASE}/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     f"{DRIVE_BASE}/clip/umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
    (f"{BASE}/split_files/vae/wan_2.1_vae.safetensors",
     f"{DRIVE_BASE}/vae/wan_2.1_vae.safetensors"),
    (f"{BASE}/split_files/clip_vision/clip_vision_h.safetensors",
     f"{DRIVE_BASE}/clip_vision/clip_vision_h.safetensors"),
    (f"{BASE}/split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors",
     f"{DRIVE_BASE}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors"),
    (f"{BASE}/split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors",
     f"{DRIVE_BASE}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors"),
    (f"https://civitai.com/api/download/models/2870639",
     f"{DRIVE_BASE}/loras/wan22_smoking_high.safetensors"),
]

# Drive の空き確認
stat = os.statvfs("/content/drive/MyDrive")
free_gb = stat.f_bavail * stat.f_frsize / 1e9
print(f"Drive 空き容量: {free_gb:.1f} GB")
if free_gb < 50:
    print("⚠️  警告: Drive の空き容量が50GB未満です（全ファイル約45GB）")
print()

# ファイルサイズ検証関数（破損ダウンロード検出）
MIN_SIZES = {
    "wan2.2_i2v_high_noise_14B_fp16.safetensors": 27.0,
    "wan2.2_i2v_low_noise_14B_fp16.safetensors":  27.0,
    "umt5_xxl_fp8_e4m3fn_scaled.safetensors":       4.5,
    "wan_2.1_vae.safetensors":                      0.4,
    "clip_vision_h.safetensors":                    0.6,
    "wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors": 0.1,
    "wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors":  0.1,
    "wan22_smoking_high.safetensors": 0.05,
}

all_ok = True
for url, dest in FILES:
    fname = os.path.basename(dest)
    min_gb = MIN_SIZES.get(fname, 0.0)
    if os.path.exists(dest):
        size_gb = os.path.getsize(dest) / 1e9
        if size_gb >= min_gb:
            print(f"✅ スキップ（Drive に既存 {size_gb:.1f}GB）: {fname}")
            continue
        else:
            print(f"⚠️  破損の可能性（{size_gb:.2f}GB < {min_gb}GB）: {fname} — 再ダウンロード")
            os.remove(dest)
    print(f"⬇️  ダウンロード中 → Drive: {fname}")
    result = subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", dest, url])
    if result.returncode == 0:
        size_gb = os.path.getsize(dest) / 1e9
        print(f"✅ 完了 ({size_gb:.1f}GB)\n")
    else:
        print(f"❌ 失敗（再実行で続きから再開できます）\n")
        all_ok = False

print()
if all_ok:
    print("✅ セル2 完了 — 全モデル準備OK")
else:
    print("⚠️  一部ダウンロード失敗。このセルを再実行してください。")

In [ ]:
# ============================================================
# セル3: ワークフロー保存
# (god_motion: 97フレーム / 6ステップ / CFG 4.0)
# ============================================================
import json, os

WORKFLOW = {
  "version": 0.4,
  "last_node_id": 20,
  "last_link_id": 32,
  "nodes": [
    {
      "id": 1, "type": "UNETLoader",
      "pos": [50, 50], "size": [340, 98], "flags": {}, "order": 0, "mode": 0,
      "inputs": [
        {"name": "unet_name", "type": "COMBO", "widget": {"name": "unet_name"}, "link": None},
        {"name": "weight_dtype", "type": "COMBO", "widget": {"name": "weight_dtype"}, "link": None}
      ],
      "outputs": [{"name": "MODEL", "type": "MODEL", "slot_index": 0, "links": [19]}],
      "properties": {"Node name for S&R": "UNETLoader"},
      "widgets_values": ["wan2.2_i2v_high_noise_14B_fp16.safetensors", "default"]
    },
    {
      "id": 16, "type": "UNETLoader",
      "pos": [50, 680], "size": [340, 98], "flags": {}, "order": 1, "mode": 0,
      "inputs": [
        {"name": "unet_name", "type": "COMBO", "widget": {"name": "unet_name"}, "link": None},
        {"name": "weight_dtype", "type": "COMBO", "widget": {"name": "weight_dtype"}, "link": None}
      ],
      "outputs": [{"name": "MODEL", "type": "MODEL", "slot_index": 0, "links": [21]}],
      "properties": {"Node name for S&R": "UNETLoader"},
      "widgets_values": ["wan2.2_i2v_low_noise_14B_fp16.safetensors", "default"]
    },
    {
      "id": 2, "type": "CLIPLoader",
      "pos": [50, 200], "size": [340, 106], "flags": {}, "order": 2, "mode": 0,
      "inputs": [
        {"name": "clip_name", "type": "COMBO", "widget": {"name": "clip_name"}, "link": None},
        {"name": "type", "type": "COMBO", "widget": {"name": "type"}, "link": None},
        {"name": "device", "shape": 7, "type": "COMBO", "widget": {"name": "device"}, "link": None}
      ],
      "outputs": [{"name": "CLIP", "type": "CLIP", "slot_index": 0, "links": [20, 22]}],
      "properties": {"Node name for S&R": "CLIPLoader"},
      "widgets_values": ["umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default"]
    },
    {
      "id": 3, "type": "CLIPVisionLoader",
      "pos": [50, 360], "size": [340, 82], "flags": {}, "order": 3, "mode": 0,
      "inputs": [{"name": "clip_name", "type": "COMBO", "widget": {"name": "clip_name"}, "link": None}],
      "outputs": [{"name": "CLIP_VISION", "type": "CLIP_VISION", "slot_index": 0, "links": [4]}],
      "properties": {"Node name for S&R": "CLIPVisionLoader"},
      "widgets_values": ["clip_vision_h.safetensors"]
    },
    {
      "id": 4, "type": "VAELoader",
      "pos": [50, 490], "size": [340, 82], "flags": {}, "order": 4, "mode": 0,
      "inputs": [{"name": "vae_name", "type": "COMBO", "widget": {"name": "vae_name"}, "link": None}],
      "outputs": [{"name": "VAE", "type": "VAE", "slot_index": 0, "links": [5, 18]}],
      "properties": {"Node name for S&R": "VAELoader"},
      "widgets_values": ["wan_2.1_vae.safetensors"]
    },
    {
      "id": 15, "type": "LoraLoader",
      "pos": [450, 80], "size": [340, 126], "flags": {}, "order": 6, "mode": 0,
      "inputs": [
        {"name": "model", "type": "MODEL", "link": 19},
        {"name": "clip", "type": "CLIP", "link": 20},
        {"name": "lora_name", "type": "COMBO", "widget": {"name": "lora_name"}, "link": None},
        {"name": "strength_model", "type": "FLOAT", "widget": {"name": "strength_model"}, "link": None},
        {"name": "strength_clip", "type": "FLOAT", "widget": {"name": "strength_clip"}, "link": None}
      ],
      "outputs": [
        {"name": "MODEL", "type": "MODEL", "slot_index": 0, "links": [28]},
        {"name": "CLIP", "type": "CLIP", "slot_index": 1, "links": [29]}
      ],
      "properties": {"Node name for S&R": "LoraLoader"},
      "widgets_values": ["wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", 1.0, 1.0]
    },
    {
      "id": 20, "type": "LoraLoader",
      "pos": [650, 80], "size": [340, 126], "flags": {}, "order": 7, "mode": 0,
      "inputs": [
        {"name": "model", "type": "MODEL", "link": 28},
        {"name": "clip", "type": "CLIP", "link": 29},
        {"name": "lora_name", "type": "COMBO", "widget": {"name": "lora_name"}, "link": None},
        {"name": "strength_model", "type": "FLOAT", "widget": {"name": "strength_model"}, "link": None},
        {"name": "strength_clip", "type": "FLOAT", "widget": {"name": "strength_clip"}, "link": None}
      ],
      "outputs": [
        {"name": "MODEL", "type": "MODEL", "slot_index": 0, "links": [30]},
        {"name": "CLIP", "type": "CLIP", "slot_index": 1, "links": [31, 32]}
      ],
      "properties": {"Node name for S&R": "LoraLoader"},
      "widgets_values": ["wan22_smoking_high.safetensors", 1.0, 1.0]
    },
    {
      "id": 17, "type": "LoraLoader",
      "pos": [450, 680], "size": [340, 126], "flags": {}, "order": 7, "mode": 0,
      "inputs": [
        {"name": "model", "type": "MODEL", "link": 21},
        {"name": "clip", "type": "CLIP", "link": 22},
        {"name": "lora_name", "type": "COMBO", "widget": {"name": "lora_name"}, "link": None},
        {"name": "strength_model", "type": "FLOAT", "widget": {"name": "strength_model"}, "link": None},
        {"name": "strength_clip", "type": "FLOAT", "widget": {"name": "strength_clip"}, "link": None}
      ],
      "outputs": [
        {"name": "MODEL", "type": "MODEL", "slot_index": 0, "links": [23]},
        {"name": "CLIP", "type": "CLIP", "slot_index": 1, "links": []}
      ],
      "properties": {"Node name for S&R": "LoraLoader"},
      "widgets_values": ["wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", 1.0, 1.0]
    },
    {
      "id": 5, "type": "LoadImage",
      "pos": [860, 50], "size": [315, 314], "flags": {}, "order": 5, "mode": 0,
      "inputs": [
        {"name": "image", "type": "COMBO", "widget": {"name": "image"}, "link": None},
        {"name": "upload", "type": "IMAGEUPLOAD", "widget": {"name": "upload"}, "link": None}
      ],
      "outputs": [
        {"name": "IMAGE", "type": "IMAGE", "slot_index": 0, "links": [6]},
        {"name": "MASK", "type": "MASK", "slot_index": 1, "links": []}
      ],
      "properties": {"Node name for S&R": "LoadImage"},
      "widgets_values": ["example.png", "image"]
    },
    {
      "id": 6, "type": "ImageScale",
      "pos": [1230, 50], "size": [315, 130], "flags": {}, "order": 12, "mode": 0,
      "inputs": [
        {"name": "image", "type": "IMAGE", "link": 6},
        {"name": "upscale_method", "type": "COMBO", "widget": {"name": "upscale_method"}, "link": None},
        {"name": "width", "type": "INT", "widget": {"name": "width"}, "link": None},
        {"name": "height", "type": "INT", "widget": {"name": "height"}, "link": None},
        {"name": "crop", "type": "COMBO", "widget": {"name": "crop"}, "link": None}
      ],
      "outputs": [{"name": "IMAGE", "type": "IMAGE", "slot_index": 0, "links": [7, 8]}],
      "properties": {"Node name for S&R": "ImageScale"},
      "widgets_values": ["lanczos", 784, 1168, "disabled"]
    },
    {
      "id": 7, "type": "CLIPVisionEncode",
      "pos": [860, 430], "size": [315, 98], "flags": {}, "order": 13, "mode": 0,
      "inputs": [
        {"name": "clip_vision", "type": "CLIP_VISION", "link": 4},
        {"name": "image", "type": "IMAGE", "link": 7},
        {"name": "crop", "type": "COMBO", "widget": {"name": "crop"}, "link": None}
      ],
      "outputs": [{"name": "CLIP_VISION_OUTPUT", "type": "CLIP_VISION_OUTPUT", "slot_index": 0, "links": [9]}],
      "properties": {"Node name for S&R": "CLIPVisionEncode"},
      "widgets_values": ["center"]
    },
    {
      "id": 8, "type": "CLIPTextEncode",
      "pos": [1230, 250], "size": [425, 220], "flags": {}, "order": 10, "mode": 0,
      "inputs": [
        {"name": "clip", "type": "CLIP", "link": 31},
        {"name": "text", "type": "STRING", "widget": {"name": "text"}, "link": None}
      ],
      "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "slot_index": 0, "links": [10]}],
      "properties": {"Node name for S&R": "CLIPTextEncode"},
      "widgets_values": ["masterpiece, best quality, cinematic action scene, dynamic camera movement, camera tracking shot, dramatic motion blur, strong parallax motion, extreme motion, fast movement, wind blowing hair and clothes, handheld camera, cinematic lighting"]
    },
    {
      "id": 9, "type": "CLIPTextEncode",
      "pos": [1230, 520], "size": [425, 220], "flags": {}, "order": 11, "mode": 0,
      "inputs": [
        {"name": "clip", "type": "CLIP", "link": 32},
        {"name": "text", "type": "STRING", "widget": {"name": "text"}, "link": None}
      ],
      "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "slot_index": 0, "links": [11]}],
      "properties": {"Node name for S&R": "CLIPTextEncode"},
      "widgets_values": ["worst quality, bad quality, blurry, distorted, watermark, nsfw, static, still frame, frozen pose, stiff, no motion"]
    },
    {
      "id": 10, "type": "WanImageToVideo",
      "pos": [1720, 200], "size": [340, 230], "flags": {}, "order": 14, "mode": 0,
      "inputs": [
        {"name": "positive", "type": "CONDITIONING", "link": 10},
        {"name": "negative", "type": "CONDITIONING", "link": 11},
        {"name": "vae", "type": "VAE", "link": 5},
        {"name": "clip_vision_output", "shape": 7, "type": "CLIP_VISION_OUTPUT", "link": 9},
        {"name": "start_image", "shape": 7, "type": "IMAGE", "link": 8},
        {"name": "width", "type": "INT", "widget": {"name": "width"}, "link": None},
        {"name": "height", "type": "INT", "widget": {"name": "height"}, "link": None},
        {"name": "length", "type": "INT", "widget": {"name": "length"}, "link": None},
        {"name": "batch_size", "type": "INT", "widget": {"name": "batch_size"}, "link": None}
      ],
      "outputs": [
        {"name": "positive", "type": "CONDITIONING", "slot_index": 0, "links": [12, 25]},
        {"name": "negative", "type": "CONDITIONING", "slot_index": 1, "links": [13, 26]},
        {"name": "latent", "type": "LATENT", "slot_index": 2, "links": [14]}
      ],
      "properties": {"Node name for S&R": "WanImageToVideo"},
      "widgets_values": [784, 1168, 97, 1]
    },
    {
      "id": 11, "type": "VideoLinearCFGGuidance",
      "pos": [1720, 480], "size": [340, 82], "flags": {}, "order": 8, "mode": 0,
      "inputs": [
        {"name": "model", "type": "MODEL", "link": 30},
        {"name": "min_cfg", "type": "FLOAT", "widget": {"name": "min_cfg"}, "link": None}
      ],
      "outputs": [{"name": "MODEL", "type": "MODEL", "slot_index": 0, "links": [15]}],
      "properties": {"Node name for S&R": "VideoLinearCFGGuidance"},
      "widgets_values": [2]
    },
    {
      "id": 18, "type": "VideoLinearCFGGuidance",
      "pos": [1720, 720], "size": [340, 82], "flags": {}, "order": 9, "mode": 0,
      "inputs": [
        {"name": "model", "type": "MODEL", "link": 23},
        {"name": "min_cfg", "type": "FLOAT", "widget": {"name": "min_cfg"}, "link": None}
      ],
      "outputs": [{"name": "MODEL", "type": "MODEL", "slot_index": 0, "links": [24]}],
      "properties": {"Node name for S&R": "VideoLinearCFGGuidance"},
      "widgets_values": [2]
    },
    {
      "id": 12, "type": "KSamplerAdvanced",
      "pos": [2120, 200], "size": [340, 310], "flags": {}, "order": 15, "mode": 0,
      "inputs": [
        {"name": "model", "type": "MODEL", "link": 15},
        {"name": "positive", "type": "CONDITIONING", "link": 12},
        {"name": "negative", "type": "CONDITIONING", "link": 13},
        {"name": "latent_image", "type": "LATENT", "link": 14},
        {"name": "add_noise", "type": "COMBO", "widget": {"name": "add_noise"}, "link": None},
        {"name": "noise_seed", "type": "INT", "widget": {"name": "noise_seed"}, "link": None},
        {"name": "steps", "type": "INT", "widget": {"name": "steps"}, "link": None},
        {"name": "cfg", "type": "FLOAT", "widget": {"name": "cfg"}, "link": None},
        {"name": "sampler_name", "type": "COMBO", "widget": {"name": "sampler_name"}, "link": None},
        {"name": "scheduler", "type": "COMBO", "widget": {"name": "scheduler"}, "link": None},
        {"name": "start_at_step", "type": "INT", "widget": {"name": "start_at_step"}, "link": None},
        {"name": "end_at_step", "type": "INT", "widget": {"name": "end_at_step"}, "link": None},
        {"name": "return_with_leftover_noise", "type": "COMBO", "widget": {"name": "return_with_leftover_noise"}, "link": None}
      ],
      "outputs": [{"name": "LATENT", "type": "LATENT", "slot_index": 0, "links": [16]}],
      "properties": {"Node name for S&R": "KSamplerAdvanced"},
      "widgets_values": ["enable", 654150814691069, "randomize", 6, 4.0, "euler", "beta", 0, 3, "enable"]
    },
    {
      "id": 19, "type": "KSamplerAdvanced",
      "pos": [2120, 580], "size": [340, 310], "flags": {}, "order": 16, "mode": 0,
      "inputs": [
        {"name": "model", "type": "MODEL", "link": 24},
        {"name": "positive", "type": "CONDITIONING", "link": 25},
        {"name": "negative", "type": "CONDITIONING", "link": 26},
        {"name": "latent_image", "type": "LATENT", "link": 16},
        {"name": "add_noise", "type": "COMBO", "widget": {"name": "add_noise"}, "link": None},
        {"name": "noise_seed", "type": "INT", "widget": {"name": "noise_seed"}, "link": None},
        {"name": "steps", "type": "INT", "widget": {"name": "steps"}, "link": None},
        {"name": "cfg", "type": "FLOAT", "widget": {"name": "cfg"}, "link": None},
        {"name": "sampler_name", "type": "COMBO", "widget": {"name": "sampler_name"}, "link": None},
        {"name": "scheduler", "type": "COMBO", "widget": {"name": "scheduler"}, "link": None},
        {"name": "start_at_step", "type": "INT", "widget": {"name": "start_at_step"}, "link": None},
        {"name": "end_at_step", "type": "INT", "widget": {"name": "end_at_step"}, "link": None},
        {"name": "return_with_leftover_noise", "type": "COMBO", "widget": {"name": "return_with_leftover_noise"}, "link": None}
      ],
      "outputs": [{"name": "LATENT", "type": "LATENT", "slot_index": 0, "links": [27]}],
      "properties": {"Node name for S&R": "KSamplerAdvanced"},
      "widgets_values": ["disable", 654150814691069, "randomize", 6, 4.0, "euler", "beta", 3, 6, "disable"]
    },
    {
      "id": 13, "type": "VAEDecodeTiled",
      "pos": [2520, 380], "size": [315, 150], "flags": {}, "order": 17, "mode": 0,
      "inputs": [
        {"name": "samples", "type": "LATENT", "link": 27},
        {"name": "vae", "type": "VAE", "link": 18},
        {"name": "tile_size", "type": "INT", "widget": {"name": "tile_size"}, "link": None},
        {"name": "overlap", "type": "INT", "widget": {"name": "overlap"}, "link": None},
        {"name": "temporal_size", "type": "INT", "widget": {"name": "temporal_size"}, "link": None},
        {"name": "temporal_overlap", "type": "INT", "widget": {"name": "temporal_overlap"}, "link": None}
      ],
      "outputs": [{"name": "IMAGE", "type": "IMAGE", "slot_index": 0, "links": [17]}],
      "properties": {"Node name for S&R": "VAEDecodeTiled"},
      "widgets_values": [512, 64, 64, 4]
    },
    {
      "id": 14, "type": "SaveAnimatedWEBP",
      "pos": [2900, 380], "size": [315, 154], "flags": {}, "order": 18, "mode": 0,
      "inputs": [
        {"name": "images", "type": "IMAGE", "link": 17},
        {"name": "filename_prefix", "type": "STRING", "widget": {"name": "filename_prefix"}, "link": None},
        {"name": "fps", "type": "FLOAT", "widget": {"name": "fps"}, "link": None},
        {"name": "lossless", "type": "BOOLEAN", "widget": {"name": "lossless"}, "link": None},
        {"name": "quality", "type": "INT", "widget": {"name": "quality"}, "link": None},
        {"name": "method", "type": "COMBO", "widget": {"name": "method"}, "link": None}
      ],
      "outputs": [],
      "properties": {},
      "widgets_values": ["ComfyUI_wan2_god_motion", 24, True, 95, "default"]
    }
  ],
  "links": [
    [28, 15, 0, 20, 0, "MODEL"],  [29, 15, 1, 20, 1, "CLIP"],
    [30, 20, 0, 11, 0, "MODEL"],  [31, 20, 1,  8, 0, "CLIP"],
    [32, 20, 1,  9, 0, "CLIP"],   [4,   3, 0,  7, 0, "CLIP_VISION"],
    [5,   4, 0, 10, 2, "VAE"],    [6,   5, 0,  6, 0, "IMAGE"],
    [7,   6, 0,  7, 1, "IMAGE"],  [8,   6, 0, 10, 4, "IMAGE"],
    [9,   7, 0, 10, 3, "CLIP_VISION_OUTPUT"],
    [10,  8, 0, 10, 0, "CONDITIONING"],
    [11,  9, 0, 10, 1, "CONDITIONING"],
    [12, 10, 0, 12, 1, "CONDITIONING"],
    [13, 10, 1, 12, 2, "CONDITIONING"],
    [14, 10, 2, 12, 3, "LATENT"],
    [15, 11, 0, 12, 0, "MODEL"],
    [16, 12, 0, 19, 3, "LATENT"],
    [17, 13, 0, 14, 0, "IMAGE"],  [18,  4, 0, 13, 1, "VAE"],
    [19,  1, 0, 15, 0, "MODEL"],  [20,  2, 0, 15, 1, "CLIP"],
    [21, 16, 0, 17, 0, "MODEL"],  [22,  2, 0, 17, 1, "CLIP"],
    [23, 17, 0, 18, 0, "MODEL"],  [24, 18, 0, 19, 0, "MODEL"],
    [25, 10, 0, 19, 1, "CONDITIONING"],
    [26, 10, 1, 19, 2, "CONDITIONING"],
    [27, 19, 0, 13, 0, "LATENT"]
  ],
  "groups": [], "config": {},
  "extra": {"ds": {"scale": 0.35, "offset": [-1.333, 100.0]}, "workflowRendererVersion": "LG"},
  "version": 0.4
}

save_path = "/content/ComfyUI/user/default/workflows/wan22_i2v_lightx2v_god_motion.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(WORKFLOW, f, ensure_ascii=False, indent=2)
print(f"✅ ワークフローを保存しました: {save_path}")
print("\n✅ セル3 完了")

In [ ]:
# ============================================================
# セル4: ComfyUI 起動
# ============================================================
import subprocess, threading, time, sys
from google.colab.output import eval_js

LOG_FILE = "/content/comfyui.log"

# VRAM 確認して起動フラグを決定
vram_result = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
vram_mb = int(vram_result.stdout.strip())
# A100(40GB)=40960, L4(24GB)=24576, T4(16GB)=16384
if vram_mb >= 38000:
    extra_flags = []  # A100: フルVRAM使用
    print(f"GPU: A100相当 ({vram_mb//1024}GB) — 標準モードで起動")
elif vram_mb >= 22000:
    extra_flags = ["--lowvram"]  # L4: lowvram
    print(f"GPU: {vram_mb//1024}GB — --lowvram モードで起動")
else:
    extra_flags = ["--lowvram", "--cpu-vae"]  # T4: lowvram + CPU VAE
    print(f"GPU: {vram_mb//1024}GB — --lowvram --cpu-vae モードで起動")
    print("⚠️  T4では97フレーム生成がOOMになる可能性があります。")
    print("   フレーム数を削減するにはワークフローのlengthを49に変更してください。")

def run_comfyui():
    cmd = [
        sys.executable, "/content/ComfyUI/main.py",
        "--listen", "0.0.0.0",
        "--port", "8188",
        "--enable-cors-header",
        "--preview-method", "auto",
        "--output-directory", "/content/drive/MyDrive/ComfyUI_models/output",
    ] + extra_flags
    with open(LOG_FILE, "w") as log:
        subprocess.run(cmd, stdout=log, stderr=log)

t = threading.Thread(target=run_comfyui, daemon=True)
t.start()
print("ComfyUI 起動中...")

for i in range(90):
    time.sleep(2)
    try:
        log = open(LOG_FILE).read()
        if "To see the GUI" in log or "Starting server" in log:
            break
        if "Error" in log and i > 10:
            # 起動エラーを早期検出
            error_lines = [l for l in log.split("\n") if "Error" in l or "error" in l]
            if error_lines:
                print(f"\n❌ 起動エラー検出:\n" + "\n".join(error_lines[-3:]))
                break
    except FileNotFoundError:
        pass
    print(".", end="", flush=True)

print("\n")

url = eval_js("google.colab.kernel.proxyPort(8188)")
print("=" * 60)
print(f"✅ ComfyUI 起動完了！")
print(f"   アクセスURL: {url}")
print("=" * 60)
print("【次の手順】")
print("1. 上記URLをブラウザで開く")
print("2. メニュー → Load → wan22_i2v_lightx2v_god_motion.json を選択")
print("3. LoadImage で画像をアップロードして Queue Prompt を実行")
print("\n生成仕様:")
print("  - 解像度: 784x1168")
print("  - フレーム数: 97フレーム (約4秒 @ 24fps)")
print("  - ステップ数: 6 (high noise 0-3 / low noise 3-6)")
print("  - CFG: 4.0")
print("\n※ このセルを止めるとComfyUIも停止します")

while True:
    time.sleep(60)